# Persian Tweet Text Generation — GPT-style Transformer Decoder


## 0. Setup

In [ ]:
import subprocess, sys, time

print('Installing p7zip...')
subprocess.run(['apt-get', 'install', '-y', 'p7zip-full'], capture_output=True)
print('Installing Python packages...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'tqdm', 'hazm'], capture_output=True)
print('All done.')

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
import os, math, time, re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from tqdm import tqdm
from hazm import Normalizer
import random

# ── Paths —─────────────────────────────────────────────────────
DRIVE_PATH  = '/content/drive/MyDrive/twitter_sample_tweets.csv.7z'
DATA_DIR    = '/content/data'
CLEANED_CSV = '/content/cleaned_tweets.csv'
DATA_PATH   = CLEANED_CSV
CKPT_DIR    = '/content/drive/MyDrive/persian_gpt_checkpoints'
MODEL_DIR   = '/content/drive/MyDrive/persian_gpt_checkpoints/final'

# ── Data — ──────────────────────────────────────────────────────
N_TWEETS   = 80_000
MAX_LENGTH = 32

# ── Transformer Decoder Hyper-parameters ────────────────────────
# ── Model architecture ──────────────────────────────────────────
EMBED_DIM  = 256      # embedding / d_model dimension
N_HEADS    = 8        # number of attention heads (EMBED_DIM must be divisible)
FF_DIM     = 512      # feed-forward hidden dim (≈ 2× EMBED_DIM)
N_LAYERS   = 4        # number of Transformer decoder layers
DROPOUT    = 0.1

# ── Training ────────────────────────────────────────────────────
BATCH_SIZE   = 32
EPOCHS       = 10
LR           = 3e-4
WARMUP_RATIO = 0.1
LABEL_SMOOTH = 0.1
GRAD_CLIP    = 1.0

random.seed(42)
torch.manual_seed(42)
print('Config loaded.')
print(f'Model: d={EMBED_DIM}  heads={N_HEADS}  ff={FF_DIM}  layers={N_LAYERS}')

## 1. Mount Drive & Extract Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

extracted_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.csv')]
if extracted_files:
    RAW_CSV = os.path.join(DATA_DIR, extracted_files[0])
    print(f'Already extracted: {RAW_CSV}')
else:
    print(f'Extracting {DRIVE_PATH} ...')
    t0 = time.time()
    result = subprocess.run(
        ['7z', 'x', DRIVE_PATH, f'-o{DATA_DIR}', '-y'],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print('STDERR:', result.stderr[-300:])
    print(f'Done in {(time.time()-t0)/60:.1f} min')
    extracted_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.csv')]
    assert extracted_files, 'No CSV found!'
    RAW_CSV = os.path.join(DATA_DIR, extracted_files[0])
    print(f'CSV: {RAW_CSV}  ({os.path.getsize(RAW_CSV)/1e9:.2f} GB)')

## 2. Detect CSV Structure

In [ ]:
PERSIAN_RE = re.compile(r'[\u0600-\u06FF]')

def persian_ratio(s):
    s = str(s)
    return len(PERSIAN_RE.findall(s)) / len(s) if s else 0.0

def detect_header_and_text_column(csv_path, n_peek=10):
    peek = pd.read_csv(csv_path, header=None, nrows=n_peek,
                       on_bad_lines='skip', encoding='utf-8', low_memory=False)
    print(peek.head(3).to_string())
    row0_ratios = [persian_ratio(peek.iloc[0, c]) for c in range(peek.shape[1])]
    data_ratios = [peek.iloc[1:, c].astype(str).apply(persian_ratio).mean()
                   for c in range(peek.shape[1])]
    has_header  = np.mean(row0_ratios) < 0.1 and np.mean(data_ratios) > 0.05
    peek_data   = peek.iloc[1:] if has_header else peek
    scores      = []
    for c in range(peek_data.shape[1]):
        col = peek_data.iloc[:, c].astype(str)
        scores.append(0.5*col.apply(persian_ratio).mean() + 0.5*col.apply(len).mean())
    text_col_idx = int(np.argmax(scores))
    print(f'Has header: {has_header}  |  Text column: {text_col_idx}')
    return has_header, text_col_idx

HAS_HEADER, TEXT_COL_IDX = detect_header_and_text_column(RAW_CSV)

## 3. Preprocessing

In [ ]:
normalizer = Normalizer()
URL_RE     = re.compile(r'https?://\S+')
MENTION_RE = re.compile(r'@\w+')
HASHTAG_RE = re.compile(r'#')
DIGIT_RE   = re.compile(r'[0-9\u06F0-\u06F9]')
KEEP_RE    = re.compile(r'[^\u0600-\u06FF .\u060C,!?\s]')
REPEAT_RE  = re.compile(r'(.)\1{2,}')
SPACE_RE   = re.compile(r'\s+')

def clean_tweet(text):
    text = str(text)
    text = URL_RE.sub('', text)
    text = MENTION_RE.sub('', text)
    text = HASHTAG_RE.sub('', text)
    text = DIGIT_RE.sub('', text)
    text = KEEP_RE.sub('', text)
    try:
        text = normalizer.normalize(text)
    except Exception:
        pass
    text = REPEAT_RE.sub(r'\1\1', text)
    return SPACE_RE.sub(' ', text).strip()

if os.path.exists(CLEANED_CSV) and os.path.getsize(CLEANED_CSV) > 1_000_000:
    print(f'Cleaned CSV exists: {CLEANED_CSV}')
else:
    print('Preprocessing...')
    seen, total_written = set(), 0
    t0 = time.time()
    csv_kw = dict(chunksize=100_000, usecols=[TEXT_COL_IDX],
                  header=0 if HAS_HEADER else None,
                  on_bad_lines='skip', encoding='utf-8', low_memory=False)
    with open(CLEANED_CSV, 'w', encoding='utf-8') as fout:
        fout.write('text\n')
        for ci, chunk in enumerate(pd.read_csv(RAW_CSV, **csv_kw)):
            if total_written >= N_TWEETS: break
            chunk.columns = ['text']
            chunk = chunk.dropna(subset=['text'])
            chunk['c'] = chunk['text'].apply(clean_tweet)
            chunk = chunk[chunk['c'].apply(lambda t: len(t.split())) >= 5]
            chunk = chunk[~chunk['c'].isin(seen)]
            seen.update(chunk['c'].tolist())
            chunk = chunk.head(N_TWEETS - total_written)
            total_written += len(chunk)
            for line in chunk['c']:
                fout.write(line + '\n')
            print(f'Chunk {ci+1} | kept={total_written:,}', end='\r')
    print(f'\nDone. {total_written:,} lines  ({(time.time()-t0)/60:.1f} min)')

## 4. Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('HooshvareLab/gpt2-fa')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

VOCAB_SIZE = len(tokenizer)
PAD_ID     = tokenizer.pad_token_id
EOS_ID     = tokenizer.eos_token_id
print(f'Vocab size: {VOCAB_SIZE:,}')
print(f'PAD id: {PAD_ID}  EOS id: {EOS_ID}')

## 5. Load & Tokenize

In [ ]:
df     = pd.read_csv(DATA_PATH, usecols=['text'], nrows=N_TWEETS)
tweets = df['text'].astype(str).dropna().tolist()
tweets = [re.sub(r'_', ' ', t) for t in tweets]
tweets = [re.sub(r'\s+', ' ', t).strip() for t in tweets]
tweets = [t for t in tweets if len(t) > 5]
print(f'Tweets: {len(tweets):,}')
print(f'Example: {tweets[0]}')

print('Tokenizing...')
t0 = time.time()
encodings = tokenizer(
    tweets,
    add_special_tokens=True,
    padding='max_length',
    truncation=True,
    max_length=MAX_LENGTH,
)
print(f'Done in {time.time()-t0:.1f}s')

## 6. Dataset & DataLoader

In [ ]:
class TweetDataset(Dataset):
    def __init__(self, encodings):
        self.input_ids = encodings['input_ids']
    def __len__(self):
        return len(self.input_ids)
    def __getitem__(self, idx):
        ids = self.input_ids[idx]
        x   = torch.tensor(ids[:-1], dtype=torch.long)  # input tokens
        y   = torch.tensor(ids[1:],  dtype=torch.long)  # shifted targets
        return x, y

dataset    = TweetDataset(encodings)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                        pin_memory=(device.type == 'cuda'), num_workers=2)
print(f'Dataset  : {len(dataset):,}')
print(f'Batches  : {len(dataloader):,}')

## 7. GPT-style Transformer Decoder Model


In [ ]:
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, n_heads, ff_dim,
                 n_layers, max_len, dropout):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.pos_emb = nn.Embedding(max_len, embed_dim)
        self.drop    = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model        = embed_dim,
            nhead          = n_heads,
            dim_feedforward= ff_dim,
            dropout        = dropout,
            activation     = 'gelu',
            batch_first    = True,
            norm_first     = True,   # Pre-LN is more stable than Post-LN)
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers          = n_layers,
            enable_nested_tensor= False,
        )
        self.ln_f = nn.LayerNorm(embed_dim)
        self.fc   = nn.Linear(embed_dim, vocab_size, bias=False)

        # weight tying: output projection shares weights with the embeddingself.fc.weight = self.tok_emb.weight

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)

    def _causal_mask(self, T, dev):
        # additive float mask: 0 = keep, -inf = block (causal)return torch.triu(torch.full((T, T), float('-inf'), device=dev), diagonal=1)

    def forward(self, x):
        B, T  = x.shape
        pos   = torch.arange(T, device=x.device).unsqueeze(0)  # positional indices broadcast over batchemb   = self.drop(self.tok_emb(x) + self.pos_emb(pos)) # (B, T, d_model)# float causal mask — same dtype as src_key_padding_mask avoids a UserWarningmask  = self._causal_mask(T, x.device)
        # float padding mask: -inf on PAD positionspad_m = torch.zeros_like(x, dtype=torch.float)
        pad_m = pad_m.masked_fill(x == PAD_ID, float('-inf'))
        out   = self.transformer(emb, mask=mask, src_key_padding_mask=pad_m)
        out   = self.ln_f(out)
        return self.fc(out)   # (B, T, vocab_size)

model = GPTLanguageModel(
    vocab_size = VOCAB_SIZE,
    embed_dim  = EMBED_DIM,
    n_heads    = N_HEADS,
    ff_dim     = FF_DIM,
    n_layers   = N_LAYERS,
    max_len    = MAX_LENGTH,
    dropout    = DROPOUT,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'GPT parameters : {n_params:,}')
print(f'Architecture   : d={EMBED_DIM}  heads={N_HEADS}  ff={FF_DIM}  layers={N_LAYERS}')

## 8. Training

- After each epoch, a checkpoint is saved to Drive
- The previous checkpoint is deleted to save space
- If the session disconnects, just re-run this cell — training resumes from the last epoch

In [ ]:
# ignore PAD positions; label smoothing reduces overconfidence
criterion    = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=LABEL_SMOOTH)
optimizer    = torch.optim.AdamW(model.parameters(), lr=LR,
                                 weight_decay=0.01, betas=(0.9, 0.95))
total_steps  = EPOCHS * len(dataloader)
warmup_steps = int(total_steps * WARMUP_RATIO)

# LR schedule: linear warmup for WARMUP_RATIO of steps, then cosine annealing
def lr_lambda(step):
    if step < warmup_steps:
        return float(step) / max(1, warmup_steps)
    progress = float(step - warmup_steps) / max(1, total_steps - warmup_steps)
    return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# ── Checkpoint helpers ────────────────────────────────────────────────────
def ckpt_path(epoch):
    return os.path.join(CKPT_DIR, f'gpt_epoch{epoch:02d}.pt')

def find_latest_ckpt():
    found = []
    for fname in os.listdir(CKPT_DIR):
        if fname.startswith('gpt_epoch') and fname.endswith('.pt'):
            try:
                ep = int(fname.replace('gpt_epoch','').replace('.pt',''))
                found.append((ep, os.path.join(CKPT_DIR, fname)))
            except ValueError:
                pass
    return sorted(found)[-1] if found else (0, None)

def save_ckpt(epoch, stats, prev_path):
    path = ckpt_path(epoch)
    torch.save({
        'epoch':           epoch,
        'model_state':     model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'losses':          stats['losses'],
        'perplexities':    stats['perplexities'],
        'top1_accs':       stats['top1_accs'],
        'top5_accs':       stats['top5_accs'],
        'epoch_times':     stats['epoch_times'],
    }, path)
    print(f'  Saved   -> {path}')
    if prev_path and os.path.exists(prev_path):
        os.remove(prev_path)
        print(f'  Deleted -> {prev_path}')
    return path

def load_ckpt():
    last_ep, path = find_latest_ckpt()
    empty = {'losses':[],'perplexities':[],'top1_accs':[],'top5_accs':[],'epoch_times':[]}
    if path is None:
        print('No checkpoint — starting from epoch 1.')
        return 0, empty, None
    print(f'Resuming from epoch {last_ep} -> {path}')
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    return last_ep, {
        'losses':       ckpt['losses'],
        'perplexities': ckpt['perplexities'],
        'top1_accs':    ckpt.get('top1_accs', []),
        'top5_accs':    ckpt.get('top5_accs', []),
        'epoch_times':  ckpt['epoch_times'],
    }, path

# ── Training loop ─────────────────────────────────────────────────────────
start_ep, stats, last_ckpt = load_ckpt()

if start_ep >= EPOCHS:
    print(f'Already fully trained ({EPOCHS} epochs).')
else:
    print('\n' + '='*60)
    print(f' Training GPT Transformer  (epochs {start_ep+1}-{EPOCHS})')
    print(f' warmup={warmup_steps}  total={total_steps}')
    print('='*60)

    for epoch in range(start_ep + 1, EPOCHS + 1):
        model.train()
        total_loss, correct1, correct5, total_tok = 0.0, 0, 0, 0
        t0   = time.time()
        loop = tqdm(dataloader, desc=f'Epoch {epoch}/{EPOCHS}', leave=True)

        for bi, (x, y) in enumerate(loop, 1):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)              # (B, T, V) — no hidden state needed
            B, T, V = logits.shape
            lf  = logits.view(-1, V)
            yf  = y.view(-1)
            loss = criterion(lf, yf)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()
            # accuracy only on non-PAD positions
            mask = yf != PAD_ID
            n    = mask.sum().item()
            if n > 0:
                correct1  += (lf.argmax(-1)[mask] == yf[mask]).sum().item()
                correct5  += (lf.topk(5,-1).indices[mask] == yf[mask].unsqueeze(1)).any(1).sum().item()
                total_tok += n
            avg = total_loss / bi
            loop.set_postfix(
                loss=f'{avg:.4f}',
                ppl=f'{math.exp(min(avg,20)):.1f}',
                top1=f'{correct1/max(total_tok,1):.3f}',
                top5=f'{correct5/max(total_tok,1):.3f}',
                lr=f'{scheduler.get_last_lr()[0]:.2e}'
            )

        elapsed  = time.time() - t0
        avg_loss = total_loss / len(dataloader)
        ppl      = math.exp(min(avg_loss, 20))
        top1     = correct1 / max(total_tok, 1)
        top5     = correct5 / max(total_tok, 1)
        stats['losses'].append(avg_loss)
        stats['perplexities'].append(ppl)
        stats['top1_accs'].append(top1)
        stats['top5_accs'].append(top5)
        stats['epoch_times'].append(elapsed / 60)

        print(f'\n=== Epoch {epoch}/{EPOCHS} ===')
        print(f'  Loss       : {avg_loss:.4f}')
        print(f'  Perplexity : {ppl:.2f}')
        print(f'  Top-1 Acc  : {top1*100:.2f}%')
        print(f'  Top-5 Acc  : {top5*100:.2f}%')
        print(f'  Time       : {elapsed/60:.1f} min')
        last_ckpt = save_ckpt(epoch, stats, last_ckpt)

    print(f'\nTotal time  : {sum(stats["epoch_times"]):.1f} min')
    print(f'Final Top-1 : {stats["top1_accs"][-1]*100:.2f}%')
    print(f'Final Top-5 : {stats["top5_accs"][-1]*100:.2f}%')

## 9. Training Curves

In [ ]:
epochs_x = list(range(1, len(stats['losses']) + 1))
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].plot(epochs_x, stats['losses'], 'o-', color='steelblue')
axes[0,0].set_title('Training Loss'); axes[0,0].set_xlabel('Epoch')
axes[0,0].set_ylabel('Cross-Entropy'); axes[0,0].grid(alpha=0.3)

axes[0,1].plot(epochs_x, stats['perplexities'], 'o-', color='tomato')
axes[0,1].set_title('Perplexity'); axes[0,1].set_xlabel('Epoch')
axes[0,1].set_ylabel('PPL'); axes[0,1].grid(alpha=0.3)

axes[1,0].plot(epochs_x, [a*100 for a in stats['top1_accs']], 'o-',
               color='seagreen', label='Top-1')
axes[1,0].plot(epochs_x, [a*100 for a in stats['top5_accs']], 's--',
               color='darkorange', label='Top-5')
axes[1,0].set_title('Accuracy'); axes[1,0].set_xlabel('Epoch')
axes[1,0].set_ylabel('Accuracy (%)'); axes[1,0].legend()
axes[1,0].grid(alpha=0.3)

axes[1,1].bar(epochs_x, stats['epoch_times'], color='steelblue', alpha=0.8)
axes[1,1].set_title('Time per Epoch'); axes[1,1].set_xlabel('Epoch')
axes[1,1].set_ylabel('Minutes'); axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/gpt_training.png', dpi=120)
plt.show()

print('\n' + '='*68)
print(f'{"Epoch":>6}  {"Loss":>8}  {"PPL":>8}  {"Top-1":>8}  {"Top-5":>8}  {"Min":>6}')
print('-'*68)
for i, ep in enumerate(epochs_x):
    print(f'{ep:>6}  {stats["losses"][i]:>8.4f}  {stats["perplexities"][i]:>8.2f}  '
          f'{stats["top1_accs"][i]*100:>7.2f}%  {stats["top5_accs"][i]*100:>7.2f}%  '
          f'{stats["epoch_times"][i]:>6.1f}')
print('='*68)

## 10. Save final model

In [ ]:
torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'model.pt'))
tokenizer.save_pretrained(os.path.join(MODEL_DIR, 'tokenizer'))
print(f'Model     -> {MODEL_DIR}/model.pt')
print(f'Tokenizer -> {MODEL_DIR}/tokenizer/')

## 11. Text Generation


In [ ]:
def generate(seed_text='', max_new_tokens=30,
             temperature=0.8, top_p=0.9, rep_penalty=1.3):
    model.eval()
    ids = tokenizer.encode(seed_text, add_special_tokens=False)
    if not ids:
        ids = [EOS_ID]

    for _ in range(max_new_tokens):
        inp    = torch.tensor([ids[-MAX_LENGTH:]], dtype=torch.long, device=device)
        with torch.no_grad():
            logits = model(inp)          # (1, T, V)
        next_logits = logits[0, -1] / temperature  # take logits at the last position# penalise tokens already in the sequence to reduce repetitionfor tok_id in set(ids):
            if next_logits[tok_id] > 0:
                next_logits[tok_id] /= rep_penalty
            else:
                next_logits[tok_id] *= rep_penalty

        # nucleus (top-p) sampling: keep the smallest set of tokens whose cumulative prob ≥ top_psorted_logits, sorted_idx = torch.sort(next_logits, descending=True)
        cum = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
        sorted_logits[cum - F.softmax(sorted_logits, dim=-1) > top_p] = float('-inf')
        next_logits = torch.zeros_like(next_logits).scatter_(0, sorted_idx, sorted_logits)
        probs   = F.softmax(next_logits, dim=-1)
        next_id = torch.multinomial(probs, 1).item()

        if next_id == EOS_ID:
            break
        ids.append(next_id)

    return tokenizer.decode(ids, skip_special_tokens=True)

PROMPTS = ['امروز', 'دانشگاه', 'ایران زیبا']
print('\n' + '='*60)
print('GENERATION RESULTS')
print('='*60)
for prompt in PROMPTS:
    print(f'\n  Prompt : {prompt}')
    print(f'  Output : {generate(prompt)}')

## 12. Summary

In [ ]:
print('='*60)
print('GPT TRANSFORMER SUMMARY')
print('='*60)
print(f'Tokenizer   : HooshvareLab/gpt2-fa  (vocab={VOCAB_SIZE:,})')
print(f'Architecture: GPT Decoder  d={EMBED_DIM}  heads={N_HEADS}  ff={FF_DIM}  layers={N_LAYERS}')
print(f'Parameters  : {n_params:,}')
print(f'Max length  : {MAX_LENGTH}')
print(f'Batch size  : {BATCH_SIZE}')
print(f'Epochs done : {len(stats["losses"])}/{EPOCHS}')
if stats['losses']:
    print(f'Final loss  : {stats["losses"][-1]:.4f}')
    print(f'Final PPL   : {stats["perplexities"][-1]:.2f}')
    print(f'Final Top-1 : {stats["top1_accs"][-1]*100:.2f}%')
    print(f'Final Top-5 : {stats["top5_accs"][-1]*100:.2f}%')
    print(f'Total time  : {sum(stats["epoch_times"]):.1f} min')
print()
print('Files on Drive:')
for p in [last_ckpt, os.path.join(MODEL_DIR, 'model.pt')]:
    if p and os.path.exists(p):
        print(f'  OK  {p}  ({os.path.getsize(p)/1e6:.1f} MB)')
print('\nDone!')